In [5]:
import os
from pypdf import PdfReader

def extract_text_from_pdf_path(pdfpath):
    text = " "
    try:
        reader = PdfReader(pdfpath)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    except Exception as e:
        print(f"Error reading {pdfpath}: {e}")
    return text


def extract_text_from_pdfs_folder_path(folderpath):
    corpus = {}
    if not os.path.exists(folderpath):
        print(f"Folder path '{folderpath}' does not exist.")
        return corpus
    for file_name in os.listdir(folderpath):
        if file_name.lower().endswith(".pdf"):
            full_path = os.path.join(folderpath,file_name)
            corpus[file_name] = extract_text_from_pdf_path(full_path)
    return corpus



print(len(extract_text_from_pdf_path("C:\\Users\\HP\\Desktop\\6ab5628d5a817_amazon_ml_challenge_problem_statement (1).pdf")))

fc = extract_text_from_pdfs_folder_path("C:\\Users\\HP\\Desktop\\resumes")

print(fc.keys())

17414
dict_keys(['amazon_ml_challenge_resume.pdf', 'Fidelity - Resume.pdf', 'fidelity_oa_revision.pdf', 'gautam resume.pdf', 'google_application_engineering_internship_resume.pdf', 'ideaclubresume.pdf', 'Resume - Sivasubramani K J.pdf', 'Resume_Aakash (1).pdf', 'Rohith_S_Amazon_Interview_final.pdf'])


In [1]:
try:
    import spacy
    print("spaCy is installed! Version:", spacy.__version__)
except ImportError:
    print("spaCy is not installed.")


spaCy is installed! Version: 3.8.16


In [4]:
import spacy
from spacy.pipeline import EntityRuler
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

samplepara = (
    "Apple is looking at buying U.K. startup for $1 billion. "
    "Elon Musk visited ISRO in Bangalore yesterday to discuss Starlink."
)

doc = nlp(samplepara)
print("----Pre-defined Entities----")
for ent in doc.ents:
    print(f"Entity: {ent.text} | Label: {ent.label_} | Meaning: {spacy.explain(ent.label_)}")

displacy.render(doc, style="ent", jupyter=True)

----Pre-defined Entities----
Entity: Apple | Label: ORG | Meaning: Companies, agencies, institutions, etc.
Entity: U.K. | Label: GPE | Meaning: Countries, cities, states
Entity: $1 billion | Label: MONEY | Meaning: Monetary values, including unit
Entity: Elon Musk | Label: PERSON | Meaning: People, including fictional
Entity: Bangalore | Label: GPE | Meaning: Countries, cities, states
Entity: yesterday | Label: DATE | Meaning: Absolute or relative dates or periods
Entity: Starlink | Label: ORG | Meaning: Companies, agencies, institutions, etc.


In [6]:
import nltk
from nltk import PCFG, CFG
from nltk.parse import ChartParser, ViterbiParser

sentence = "I saw the man with a telescope".split()

cfg_grammar = CFG.fromstring(
    """
S -> NP VP
NP -> 'I' | Det N | Det NP PP
PP -> P NP
Det -> 'the' | 'a' 
VP -> V NP | V NP PP
N -> 'man' | 'telescope' 
V -> 'saw' 
P -> 'with' 
"""
)

parser = ChartParser(cfg_grammar)
trees = list(parser.parse(sentence))

print(f"--- CFG Parsing ---")
print(f"Number of parse trees found: {len(trees)}")

for i, tree in enumerate(trees, 1):
    print(f"\nParse Trees {i}")
    tree.pretty_print()



if len(trees) > 1:
    print("\n--- Ambiguity Detected! Switching to PCFG ---")

    # Probabilities for rules under each non-terminal must sum to 1.0
    pcfg_grammar = PCFG.fromstring(
        """
    S -> NP VP [1.0]
    PP -> P NP [1.0]
    NP -> 'I' [0.2] | Det N [0.5] | Det N PP [0.3]
    VP -> V NP [0.6] | V NP PP [0.4]
    Det -> 'the' [0.6] | 'a' [0.4]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
  """
    )

    pcfg_parser = ViterbiParser(pcfg_grammar)
    most_probable_tree = list(pcfg_parser.parse(sentence))

    for tree in most_probable_tree:
        print(f"\nMost Probable Tree (Probability = {tree.prob():.6f}):")
        tree.pretty_print()

--- CFG Parsing ---
Number of parse trees found: 1

Parse Trees 1
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|________                 
 |   |       |            PP              
 |   |       |        ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw the     man with  a      telescope



In [8]:
import spacy
nlp = spacy.load("en_core_web_sm")
sentence = "The quick brown fox jumps over the lazy dog."
doc = nlp(sentence)

print("--- Dependency Parse Output ---")
print(
    f"{'Token':<12} | {'POS Tag':<8} | {'Dependency':<12} | {'Head Token':<12} | {'Children'}"
)
print("-" * 65)

for token in doc:
    children = [child.text for child in token.children]
    print(
        f"{token.text:<12} | {token.pos_:<8} | {token.dep_:<12} | {token.head.text:<12} | {children}"
    )
spacy.displacy.render(doc,style="dep", jupyter=True)

--- Dependency Parse Output ---
Token        | POS Tag  | Dependency   | Head Token   | Children
-----------------------------------------------------------------
The          | DET      | det          | fox          | []
quick        | ADJ      | amod         | fox          | []
brown        | ADJ      | amod         | fox          | []
fox          | NOUN     | nsubj        | jumps        | ['The', 'quick', 'brown']
jumps        | VERB     | ROOT         | jumps        | ['fox', 'over', '.']
over         | ADP      | prep         | jumps        | ['dog']
the          | DET      | det          | dog          | []
lazy         | ADJ      | amod         | dog          | []
dog          | NOUN     | pobj         | over         | ['the', 'lazy']
.            | PUNCT    | punct        | jumps        | []
